# 01 · Your first dbt run

Welcome! This notebook walks the whole stack end to end:

```
generate_data.py  ──▶  raw schema  ──▶  dbt  ──▶  staging / marts schemas
   (the "EL")        (PostgreSQL)      (the "T")       (your models)
```

Two ways to drive dbt from a notebook, both used below:

| interface | good for |
|---|---|
| `!dbt ...` shell cells | seeing exactly what the CLI prints |
| `dbtRunner` (Python API) | inspecting results programmatically, asserting outcomes |

Companion reading: [docs/01](../docs/01_what_is_dbt.md) and [docs/02](../docs/02_environment_setup.md).

In [ ]:
# --- Standard setup (identical in every notebook) ---------------------------
import os, sys, json, subprocess
from pathlib import Path
from datetime import date, timedelta

from dotenv import load_dotenv

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
load_dotenv(REPO / ".env")

os.chdir(REPO / "jaffle_shop")          # dbt must run from the project dir
os.environ.setdefault("DBT_PROFILES_DIR", str(REPO / "jaffle_shop"))

from dbt.cli.main import dbtRunner

def run_dbt(args):
    """Run dbt programmatically (same engine as the CLI). Returns a
    dbtRunnerResult: .success says how it went, .result has per-node details.
    Crucially it never sys.exit()s -- perfect for notebooks."""
    print(f"$ dbt {' '.join(args)}")
    res = dbtRunner().invoke(args)
    print(f"-> success={res.success}")
    return res

def load_day(*flags):
    """Invoke the raw-data generator (plays the role of the EL tool)."""
    subprocess.run(
        [sys.executable, str(REPO / "scripts" / "generate_data.py"), *flags],
        check=True,
    )


In [ ]:
# --- Warehouse connection for %%sql cells (jupysql) and pandas --------------
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DBT_USER', 'dbt')}:{os.getenv('DBT_PASSWORD', 'dbt')}"
    f"@{os.getenv('DBT_HOST', 'localhost')}:{os.getenv('DBT_PORT', '5432')}"
    f"/{os.getenv('DBT_DBNAME', 'jaffle_shop')}"
)

%load_ext sql
%sql engine
%config SqlMagic.displaylimit = 25


## 1. Make sure raw data exists

The generator is idempotent -- if days 1-2 are already loaded it does
nothing. dbt *never* loads data; this script plays the role of Fivetran/
Airbyte landing source tables in the warehouse.

In [ ]:
load_day("--days", "2")

## 2. Look at what the "EL tool" landed (this is dbt's *input*)

In [ ]:
%%sql
select table_name,
       (xpath('/row/cnt/text()',
              query_to_xml('select count(*) as cnt from raw.' || table_name,
                           false, true, '')))[1]::text::int as rows
from information_schema.tables
where table_schema = 'raw'
order by table_name

In [ ]:
%%sql
select id, customer_id, status, ordered_at, updated_at, _synced_at, _batch_day
from raw.raw_orders
order by id
limit 5

Note the columns `_synced_at` / `_batch_day`: EL-tool metadata, and the
slightly messy reality (`raw.raw_customers.first_name` casing, SHOUTING
payment methods...) that staging models will clean up.

## 3. `dbt debug` -- is everything wired up?

Checks the project files, the profile, and the warehouse connection.

In [ ]:
!dbt debug

## 4. `dbt deps` -- install packages

Installs the packages from `packages.yml` into `dbt_packages/` (gitignored),
pinned by `package-lock.yml` (committed).

In [ ]:
!dbt deps

## 5. `dbt build` -- the everything command

`build` runs **seeds + models + tests + snapshots** in DAG order. A node
only runs after its parents succeed, and a model is skipped if its
upstream tests fail -- tests *gate* the build.

In [ ]:
!dbt build

## 6. The same thing, programmatically

`dbtRunner` is the exact engine behind the CLI. The result object tells you
what happened per node -- this is how the later notebooks assert outcomes.

In [ ]:
res = run_dbt(["build", "--select", "stg_orders"])

assert res.success
for r in res.result.results:
    print(f"{r.node.name:<30} {r.status:<8} {r.execution_time:.2f}s")

## 7. What did dbt create in the warehouse?

In [ ]:
%%sql
select table_schema, table_name, table_type
from information_schema.tables
where table_schema like 'dev%'
order by table_schema, table_name

The `dev_` prefixes come from this project's `generate_schema_name` macro:
in the `dev` target every custom schema is namespaced (`dev_staging`,
`dev_marts`); in `prod` they would be clean (`staging`, `marts`).

## 8. Query a mart like an analyst would

In [ ]:
%%sql
select order_date, orders_count, revenue
from dev_marts.agg_daily_revenue
order by order_date

## Where to click next

* **Adminer** (browse the DB): [http://localhost:8080](http://localhost:8080) -- server `postgres`, user/pass/db `dbt`/`dbt`/`jaffle_shop`
* **dbt docs site**: run `make docs` in a terminal, then open [http://localhost:8081](http://localhost:8081)
* Notebook [02 · materializations](02_models_materializations.ipynb)